In [1]:
"""
基于本地 MinerU 解析 PDF + 通义千问 Embedding + Chroma 向量库的轻量化 RAG 知识库完整实现。
"""
import glob                          # 文件通配符匹配，目的匹配到目标文件
import os                            # 操作系统接口
import json                          # JSON数据处理
import subprocess                    # 运行子进程
from datetime import datetime        # 日期时间处理
import chromadb                      # 向量数据库
import torch                         # PyTorch深度学习
import doclayout_yolo.nn.tasks       # MinerU的YOLO模型相关
import dashscope                     # 阿里云灵积API
from http import HTTPStatus          # HTTP状态码
# 添加安全反序列化类，确保加载YOLO模型时不会报错
torch.serialization.add_safe_globals([doclayout_yolo.nn.tasks.YOLOv10DetectionModel])
from langchain_community.embeddings.dashscope import DashScopeEmbeddings               # LangChain的DashScope嵌入
from mineru.cli.common import do_parse, read_fn                                        # MinerU的文档解析工具
from pathlib import Path                                                               # 路径处理
from dotenv import load_dotenv                                                         # 环境变量加载
load_dotenv()

E:\Agent\L2阶段\langchain_tutorial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False

### MinerURAGSystem

In [2]:
# 内容提取相关，分块，存储，检索
class MinerURAGSystem:
    """
    整体流程：PDF文件 → MinerU解析 → json结构化内容 → 自定义文本/表格切块 → DashScope向量化 → Chroma持久化存储 → 向量检索
    """
    def __init__(self, persist_directory="./chroma_db"):
        # self.embedding_model = DashScopeEmbeddings(
        #     model=os.getenv("EMBEDDING_MODEL", "text-embedding-v3"),
        #     dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"),
        # )
        from langchain_ollama import OllamaEmbeddings
        self.embedding_model = OllamaEmbeddings(model="bge-m3")
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.collection = self.client.get_or_create_collection("documents")

    # 加载文档
    def process_documents(self, pdf_directory, output_dir="./my_mineru_processed"):
        """
        批量处理指定目录下的 PDF 文件，将其解析为结构化数据（JSON）。
        适用于 RAG 知识库的离线数据预处理阶段。
        """
        # 1. 确保输出根目录存在，不存在则递归创建
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        processed_files = []  # 用于收集成功解析的 JSON 文件路径

        # 2. 遍历 PDF 目录下的所有文件
        for filename in os.listdir(pdf_directory):
            if filename.endswith('.pdf'): # filename = xxx.pdf
                input_path = os.path.join(pdf_directory, filename) # ./my_pdfs/xxx.pdf
                output_path = os.path.join(output_dir, filename.replace('.pdf', '')) # ./my_mineru_processed/xxx
                pdf_file_name = Path(input_path).stem # xxx
                # 3. 读取 PDF 文件的二进制内容
                pdf_bytes = read_fn(input_path)
                # 4. 调用核心解析引擎
                do_parse(
                    output_dir=output_path,              # 解析结果的输出根目录
                    pdf_file_names=[pdf_file_name],      # 待解析的文件名列表（支持批量）
                    pdf_bytes_list=[pdf_bytes],          # 对应的文件二进制流列表
                    p_lang_list=["ch"],                  # 指定解析语言为中文
                    backend="pipeline"                   # 指定解析后端, 使用管道模式进行解析
                )

                # 5. 定位解析后生成的标准 JSON 文件
                content_file = os.path.join(output_path, "auto", f"{pdf_file_name}_content_list.json") # ./my_mineru_processed/xxx/auto/xxx_content_list.json

                # 6. 校验文件是否成功生成，并记录结果
                if os.path.exists(content_file):
                    processed_files.append(content_file)

        # 7. 返回所有成功处理的 JSON 文件绝对路径列表
        return processed_files

    # 切块——  数据，分块大小
    def _split_into_chunks(self, content_data, chunk_size, overlap):
        chunks = []
        current_chunk = []
        current_length = 0
        #  提前文档数据： 加了一点判断-- 标题 非标题的存储
        for item in content_data:
            if item.get('type') == 'text':
                text = item.get('text', '')
                if not text:
                    continue
                if item.get('text_level', 0) > 0:
                    if current_chunk:
                        chunks.append(
                            {'text': ' '.join(current_chunk), 'metadata': {'chunk_type': 'text', 'has_title': True}})
                    current_chunk = [f"## {text}"]
                    current_length = len(text)
                elif current_length + len(text) > chunk_size and current_chunk:
                    chunks.append({'text': ' '.join(current_chunk), 'metadata': {'chunk_type': 'text'}})
                    current_chunk = [text]
                    current_length = len(text)
                else:
                    current_chunk.append(text)
                    current_length += len(text)

            elif item.get('type') == 'table':
                if current_chunk:
                    chunks.append({'text': ' '.join(current_chunk), 'metadata': {'chunk_type': 'text'}})
                    current_chunk = []
                    current_length = 0

                table_body = item.get('table_body', '')
                if table_body:
                    if len(table_body) <= chunk_size:
                        chunks.append({
                            'text': table_body,
                            'metadata': {
                                'chunk_type': 'table',
                                'page_idx': item.get('page_idx', 0),
                                'has_caption': len(item.get('table_caption', [])) > 0,
                                'has_footnote': len(item.get('table_footnote', [])) > 0
                            }
                        })
                    else:
                        table_chunks = self._split_table_into_chunks(table_body, chunk_size)
                        for i, chunk in enumerate(table_chunks):
                            chunks.append({
                                'text': chunk,
                                'metadata': {
                                    'chunk_type': 'table',
                                    'page_idx': item.get('page_idx', 0),
                                    'table_chunk_index': i,
                                    'total_table_chunks': len(table_chunks)
                                }
                            })
        if current_chunk:
            chunks.append({'text': ' '.join(current_chunk), 'metadata': {'chunk_type': 'text'}})
        return chunks

    # 表格切块
    def _split_table_into_chunks(self, table_html, chunk_size):
        chunks = []
        if table_html.startswith('<table>'):
            table_content = table_html[7:-8]
        else:
            table_content = table_html
        rows = table_content.split('<tr>')
        table_head = rows[1]
        current_chunk = '<table>' + table_head
        print('表头:', current_chunk)

        current_chunk_size = len(current_chunk)
        for row in rows[2:]:
            row_content = '<tr>' + row
            if current_chunk_size + len(row_content) > chunk_size and current_chunk != '<table>':
                current_chunk += '</table>'
                chunks.append(current_chunk)
                current_chunk = '<table>' + table_head + row_content
                current_chunk_size = len(current_chunk)
            else:
                current_chunk += row_content
                current_chunk_size += len(row_content)

        if current_chunk != '<table>' + table_head:
            current_chunk += '</table>'
            chunks.append(current_chunk)
        return chunks

    # 向量化
    def chunk_and_embed(self, content_files, chunk_size=512, overlap=50):
        documents = []
        metadatas = []
        ids = []
        for file_path in content_files:
            with open(file_path, 'r', encoding='utf-8') as f:
                content_data = json.load(f)
            doc_metadata = {
                'source': os.path.basename(file_path),
                'file_type': 'pdf',
                'processing_date': datetime.now().isoformat()
            }
            text_chunks = self._split_into_chunks(content_data, chunk_size, overlap)
            for i, chunk in enumerate(text_chunks):
                documents.append(chunk['text'])
                metadatas.append({**doc_metadata, **chunk['metadata']})
                ids.append(f"{os.path.basename(file_path)}_chunk_{i}")
        return documents, metadatas, ids

    # 存储
    def build_vector_store(self, documents, metadatas, ids):
        if self.collection.count() > 0:
            print(f"集合不为空，正在删除 {self.collection.count()} 个旧文档...")
            existing_ids = self.collection.get(include=[])['ids']
            self.collection.delete(ids=existing_ids)
            print("旧文档删除完毕。")
        else:
            print("集合为空，无需删除。")

        embeddings = self.embedding_model.embed_documents(documents)
        self.collection.add(
            embeddings=embeddings,
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )
        print(f"成功存储 {len(documents)} 个文档块到向量数据库")

    # 检索
    def query_documents(self, query_text, n_results=5):
        query_embedding = self.embedding_model.embed_documents([query_text])[0]
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            include=['documents', 'metadatas', 'distances']
        )
        return results

In [3]:
# 集成功能完善的 RAGQASystem
class RAGQASystem:
    def __init__(self, mineru_rag_system):
        self.rag_system = mineru_rag_system
        dashscope.api_key = os.getenv("DASHSCOPE_API_KEY")

    def _build_prompt(self, question, context_docs):
        context_text = "\n\n".join([f"文档片段 {i + 1}:\n{doc}" for i, doc in enumerate(context_docs)])
        prompt = f"""请你扮演一个专业的财务报告分析师。
                    根据下面提供的几段从《厦门灿坤实业股份有限公司2019年年度报告》中摘录的文字，
                    严谨且仅依据这些信息来回答用户的问题。
                    如果提供的信息不足以回答问题，请明确告知"根据现有信息无法回答"。

                    ---
                    相关文档内容：
                    {context_text}
                    ---

                    用户问题：{question}
                    """
        return prompt

    def _call_llm(self, prompt):
        print("\n--- 正在向DashScope LLM发送请求... ---")

        response = dashscope.Generation.call(
            model='qwen-turbo',
            messages=[{'role': 'user', 'content': prompt}],
            result_format='message'
        )

        if response.status_code == HTTPStatus.OK:
            answer = response.output.choices[0]['message']['content']
            print("--- LLM响应成功 ---")
            return answer
        else:
            print(f"请求失败：request_id={response.request_id}, status_code={response.status_code}, "
                  f"code={response.code}, message={response.message}")
            return "抱歉，调用大语言模型时出错，无法生成答案。"

    def generate_answer(self, question, context):
        prompt = self._build_prompt(question, context)
        print("\n--- 发送给LLM的最终Prompt ---\n", prompt)
        answer = self._call_llm(prompt)
        return answer, context

    def ask_question(self, question):
        results = self.rag_system.query_documents(question)
        if not results or not results.get('documents') or not results['documents'][0]:
            return "抱歉，没有找到相关的文档信息。", []

        answer, context = self.generate_answer(question, results['documents'][0])
        return answer, results

In [4]:
# ==============================================================================
# --- 主调用流程--- 判断文档和存储的文件是否存在，如果不存在就解析，如果存在就跳过
# ==============================================================================

# 1. 初始化RAG系统
rag_system = MinerURAGSystem(persist_directory="./my_chroma_db")

# 2. 定义路径
pdf_folder = 'my_pdfs'
processed_base_dir = 'my_mineru_processed'

# 3. 检查是否有PDF文件可处理
if not os.path.exists(pdf_folder) or len([f for f in os.listdir(pdf_folder) if f.endswith('.pdf')]) == 0:
    print(f"请在当前目录下创建'{pdf_folder}'文件夹并放入PDF文件。")
    if not os.path.exists(pdf_folder):
        os.makedirs(pdf_folder)
    exit(0)  # 退出，等待用户放入PDF

# 4. 查找已处理的文件
processed_files = []
if os.path.exists(processed_base_dir):
    # 查找所有_content_list.json文件
    pattern = os.path.join(processed_base_dir, '**', '*_content_list.json')
    processed_files = glob.glob(pattern, recursive=True)
    processed_files = [f for f in processed_files if f]

In [ ]:
# 5. 如果没有已处理的文件，则解析PDF
if not processed_files:
    print("未找到已处理的文件，开始解析PDF...")
    processed_files = rag_system.process_documents(pdf_directory=pdf_folder)
    print(f"解析完成，生成了 {len(processed_files)} 个内容文件。")
else:
    print(f"找到 {len(processed_files)} 个已处理的文件，跳过解析步骤。")

# 6. 文本分块与向量化准备
print(f"\n开始对 {len(processed_files)} 个文件进行文本分块与向量化...")
documents, metadatas, ids = rag_system.chunk_and_embed(content_files=processed_files)
print(f"已将文档分割成 {len(documents)} 个块。")

# 7. 构建并填充向量数据库
if documents:
    print("\n开始构建并填充向量数据库...")
    rag_system.build_vector_store(documents, metadatas, ids)

    # 8. 初始化问答系统
    qa_system = RAGQASystem(mineru_rag_system=rag_system)
    print("\n问答系统已准备就绪。")

    # 9. 提出问题
    question = "2019年一季度归属于上市公司股东的净利润？"
    # question = "2019年度归属于上市公司股东的净利润？"

    print(f"\n--- 正在查询问题 --- \n{question}")
    answer, search_results = qa_system.ask_question(question)

    print("\n\n==================== 问答结果 ====================")
    print("--- 最终回答 ---")
    print(answer)
    print("\n--- 检索到的相关信息 ---")
    if search_results and search_results.get('documents') and search_results['documents'][0]:
        for i, doc in enumerate(search_results['documents'][0]):
            distance = search_results['distances'][0][i]
            metadata = search_results['metadatas'][0][i]
            print(f"\n--- 相关片段 {i + 1} (距离: {distance:.4f}) ---")
            print(f"来源: {metadata.get('source')}")
            print("内容:")
            print(doc)
    else:
        print("没有找到相关的文档片段。")
    print("\n==================================================")
else:
    print("错误：没有成功生成文档块，请检查PDF文件和处理过程。")

"""
构建RAG的检索：
1.pdf文件中的表格进行处理之后再去分块存储
2.构建rag的检索--- 自定义类的方式去实现--- 没有用langchain去实现
下课之后：作业： 构建rag的检索--- 用一下langchain去实现

"""